## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [1]:
import min_features, daily_return
import importlib
import pandas as pd

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 3, 5]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = [
    c for c in df_daily.iloc[:, 1:].columns
    if "return" not in c.lower()
]
min_cols = df_min.iloc[:, 1:].columns.to_list()

/Users/brettchase/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score

column_sets = [daily_cols]#, min_cols, daily_cols + min_cols]
names = ['daily']#, 'minute', 'daily+minute']
returns = [1]#, 3, 5]
runs = 5
test_size = 5
lbs = [6]
offset_size = test_size

models = {
    "logistic": LogisticRegression(max_iter=1000),
    "linear_svm": LinearSVC(),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

tscv = TimeSeriesSplit(n_splits=5)

results = []

for feature_cols, name in zip(column_sets, names):

    X = df_main[feature_cols].to_numpy()

    for r in returns:
        
        y = df_main[f"Return_{r}"].to_numpy()
        
        for i in range(runs): # number of runs to do

            offset = max(i * offset_size, 0) # step size for each run
            
            for lb in lbs:
                 
                df_ph = df_main.iloc[offset : offset + 245 * lb, :].copy()  # 245 records is ~1 year of data
                ret_col = f"Return_{r}"
                ret_pct_col = f"Return%_{r}"

                rets = df_ph[ret_pct_col]
                neg, pos = rets[rets < 0], rets[rets > 0]

                neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
                pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()
                filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()
                print(f"Run {i+1} of {runs} | LB: {lb} | Horizon: {r} | {filtered['Date'].iloc[test_size]} - {filtered['Date'].iloc[0]}")

                df_indicators = filtered[feature_cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = filtered[ret_col]


Run 1 of 5 | LB: 6 | Horizon: 1 | 2025-12-12 - 2025-12-19
Run 2 of 5 | LB: 6 | Horizon: 1 | 2025-12-04 - 2025-12-12
Run 3 of 5 | LB: 6 | Horizon: 1 | 2025-11-25 - 2025-12-05
Run 4 of 5 | LB: 6 | Horizon: 1 | 2025-11-19 - 2025-11-26
Run 5 of 5 | LB: 6 | Horizon: 1 | 2025-11-10 - 2025-11-19


In [20]:
df_main[daily_cols + [f"Return_{r}"]].corr()[f"Return_{r}"].sort_values()

UpVolume             -0.049139
Vol_Ratio_10         -0.047178
Max_240_Rows_Since   -0.046843
Close_Rel_Min10      -0.037971
Max_120_Rows_Since   -0.033759
                        ...   
Rel_Max_120           0.039439
100_SMA_200           0.040218
Close_Rel_Max200      0.047451
Rel_Max_240           0.052938
Return_1              1.000000
Name: Return_1, Length: 81, dtype: float64

In [29]:
def eval_models_timeseries_cv(df_window, feature_cols, target_col, n_splits=5):
    X = df_window[feature_cols].to_numpy()
    y = df_window[target_col].to_numpy()

    # normalize {-1,1} -> {0,1} if needed
    u = np.unique(y[~pd.isna(y)])
    if set(u.tolist()) == {0, 1}:
        y = (y > 0).astype(int)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    rows = []
    for model_name, model in models.items():
        bal_accs, f1s = [], []

        # validation-only distribution trackers
        val_ns, val_pos_fracs, val_pos_ns, val_neg_ns = [], [], [], []

        for tr_idx, va_idx in tscv.split(X):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model.fit(X_tr, y_tr)
            preds = model.predict(X_va)

            bal_accs.append(balanced_accuracy_score(y_va, preds))
            f1s.append(f1_score(y_va, preds, zero_division=0))

            n_va = len(y_va)
            n_pos = int((y_va == 1).sum())
            n_neg = int((y_va == 0).sum())

            val_ns.append(n_va)
            val_pos_ns.append(n_pos)
            val_neg_ns.append(n_neg)
            val_pos_fracs.append(n_pos / n_va if n_va else np.nan)

        rows.append({
            "model": model_name,
            "bal_acc_mean": float(np.mean(bal_accs)),
            "bal_acc_std": float(np.std(bal_accs)),
            "f1_mean": float(np.mean(f1s)),
            "f1_std": float(np.std(f1s)),

            # validation-only distribution summary across folds
            "val_n_mean": float(np.mean(val_ns)),
            "val_pos_frac_mean": float(np.nanmean(val_pos_fracs)),
            "val_pos_n_mean": float(np.mean(val_pos_ns)),
            "val_neg_n_mean": float(np.mean(val_neg_ns)),
            "val_pos_frac_min": float(np.nanmin(val_pos_fracs)),
            "val_pos_frac_max": float(np.nanmax(val_pos_fracs)),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False)

column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ['daily', 'minute', 'daily+minute']
returns = [1]#, 3, 5]
runs = 2
test_size = 5
lbs = [6]
offset_size = test_size

models = {
    "logistic": LogisticRegression(max_iter=1000, random_state=42),
    "linear_svm": LinearSVC(dual=False, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1,),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

tscv = TimeSeriesSplit(n_splits=5)

results = []
results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        ret_col = f"Return_{r}"
        ret_pct_col = f"Return%_{r}"

        for i in range(runs):
            offset = max(i * offset_size, 0)

            for lb in lbs:
                df_ph = df_main.iloc[offset : offset + 245 * lb, :].copy()

                rets = df_ph[ret_pct_col]
                neg, pos = rets[rets < 0], rets[rets > 0]
                neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
                pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()

                filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()
                #filtered = df_ph.copy()
                
                # Basic cleaning consistent with your pipeline
                filtered[feature_cols] = filtered[feature_cols].replace([np.inf, -np.inf], 0)

                print(
                    f"Run {i+1}/{runs} | Feats:{feat_name} | LB:{lb} | H:{r} | "
                    f"{filtered['Date'].iloc[test_size]} - {filtered['Date'].iloc[0]}"
                )

                fold_scores = eval_models_timeseries_cv(
                    df_window=filtered,
                    feature_cols=feature_cols,
                    target_col=ret_col,
                    n_splits=5,
                )

                # attach metadata so you can compare across pipeline dimensions
                fold_scores["feature_set"] = feat_name
                fold_scores["horizon"] = r
                fold_scores["run"] = i
                fold_scores["lb_years"] = lb
                fold_scores["start_date"] = filtered["Date"].iloc[0]
                fold_scores["end_date"] = filtered["Date"].iloc[-1]
                fold_scores["n_rows"] = len(filtered)

                results_all.append(fold_scores)

results_df = pd.concat(results_all, ignore_index=True)

Run 1/2 | Feats:daily | LB:6 | H:1 | 2025-12-12 - 2025-12-19
Run 2/2 | Feats:daily | LB:6 | H:1 | 2025-12-04 - 2025-12-12
Run 1/2 | Feats:minute | LB:6 | H:1 | 2025-12-12 - 2025-12-19
Run 2/2 | Feats:minute | LB:6 | H:1 | 2025-12-04 - 2025-12-12
Run 1/2 | Feats:daily+minute | LB:6 | H:1 | 2025-12-12 - 2025-12-19
Run 2/2 | Feats:daily+minute | LB:6 | H:1 | 2025-12-04 - 2025-12-12


In [15]:
filtered

,Date,early_close_%_up_minutes,early_open_%_up_minutes,early_post_market_%_up_minutes,early_pre_market_%_up_minutes,late_close_%_up_minutes,late_day_%_up_minutes,late_open_%_up_minutes,late_post_market_%_up_minutes,late_pre_market_%_up_minutes,...,Vol_Spike_10,Vol_Spike_20,Vol_Spike_40,ADL,VIX,VIX_rolling_std,VIX_crossover,VIX_5_change,VIX_1_change,VIX_10_change
1884,2025-11-19,0.462,0.667,0.500,0.500,0.500,0.537,0.846,0.537,0.367,...,1.019,1.166,1.225,0.763794,29.299999,2.0,1.0,0.271,-0.040,0.296
1883,2025-11-18,0.429,0.267,0.500,0.477,0.250,0.482,0.333,0.389,0.542,...,1.233,1.354,1.460,0.868067,30.510000,2.8,1.0,0.346,0.083,0.309
1882,2025-11-17,0.400,0.571,0.700,0.485,0.667,0.432,0.615,0.308,0.364,...,1.000,1.041,1.119,0.981575,28.160000,2.3,1.0,0.245,0.077,0.296
1881,2025-11-14,0.333,0.500,0.636,0.447,0.375,0.519,0.714,0.464,0.444,...,1.338,1.368,1.417,1.255530,26.150000,1.8,1.0,0.064,0.010,0.189
1879,2025-11-12,0.583,0.231,0.444,0.494,0.400,0.486,0.467,0.524,0.286,...,0.807,0.810,0.839,1.387293,23.059999,1.0,-1.0,0.020,0.017,0.060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,2020-01-09,0.615,0.467,0.667,0.392,0.538,0.508,0.364,0.500,0.500,...,1.071,1.083,1.297,1.852219,16.040001,0.5,1.0,0.003,-0.053,0.059
419,2020-01-08,0.333,0.769,0.875,0.500,0.400,0.712,0.533,0.727,0.417,...,1.214,1.215,1.466,1.757401,16.930000,0.5,1.0,0.002,-0.013,0.117
418,2020-01-07,0.400,0.357,0.545,0.696,0.357,0.621,0.667,0.385,0.750,...,1.019,1.031,1.239,1.711114,17.150000,0.5,1.0,-0.042,-0.002,0.185
416,2020-01-03,0.571,0.643,0.455,0.554,0.500,0.500,0.357,0.333,0.333,...,1.376,1.489,1.542,1.546401,16.950001,0.7,1.0,0.125,0.059,0.149


In [9]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
import warnings
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", message="RuntimeWarning: divide by zero encountered in matmul raw_prediction = X @ weights + intercept")

# -----------------------------
# Models (keep as you have)
# -----------------------------
models = {
    "logistic": LogisticRegression(max_iter=1000, random_state=42),
    "linear_svm": LinearSVC(dual=False, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

# -----------------------------
# Helpers
# -----------------------------
def _to_binary(y):
    """Return y as {0,1} with minimal assumptions."""
    y = np.asarray(y)
    u = np.unique(y[~pd.isna(y)])
    if set(u.tolist()) == {-1, 1}:
        return (y > 0).astype(np.int8)
    if set(u.tolist()) == {0, 1}:
        return y.astype(np.int8)
    # If already boolean
    if set(u.tolist()) == {False, True}:
        return y.astype(np.int8)
    raise ValueError(f"Unexpected target values: {u[:10]}")

def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    y_all = _to_binary(dfw[target_col].to_numpy())
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        # Purge at boundary so train labels/features don't overlap test horizon
        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
        f"Run {k+1}/{runs} | "
        f"Train: {dates[train_start]} → {dates[train_end-1]} | "
        f"Test: {dates[test_start]} → {dates[test_end-1]} | "
        f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        y_test = y_all[test_start:test_end]

        # OOT distribution (test only)
        dist = _compute_dist(y_test)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)
            preds = m.predict(X_test)

            rows.append({
                "run": k + 1,
                "model": model_name,
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "f1": float(f1_score(y_test, preds, zero_division=0)),
                "acc": float(accuracy_score(y_test, preds)),
                **dist,
                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "purge_days": purge,
                "train_years": train_years,
                "test_days": test_days,
                "step_days": step_days,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1, 3, 5]  # add 3,5,etc later
train_years_grid = [3, 5, 7]  # could be [3,4,5,6]
runs = 30
test_days = 5
step_days = 5

results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=r,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results_all.append(df_scores)

results_df = pd.concat(results_all, ignore_index=True)

# -----------------------------
# Simple summaries you’ll actually use
# -----------------------------
# mean/std across the 20 OOT runs (deployment-aligned)
summary_df = (
    results_df
    .groupby(["feature_set", "horizon", "train_years", "model"], as_index=False)
    .agg(
        bal_acc_mean=("bal_acc", "mean"),
        bal_acc_std=("bal_acc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        test_pos_frac_mean=("test_pos_frac", "mean"),
        test_pos_frac_min=("test_pos_frac", "min"),
        test_pos_frac_max=("test_pos_frac", "max"),
        n_runs=("run", "nunique"),
    )
    .sort_values(["feature_set", "horizon", "bal_acc_mean"], ascending=[True, True, False])
)

# best model per feature_set/horizon/train_years
best_df = summary_df.groupby(["feature_set", "horizon", "train_years"], as_index=False).head(1)


Run 1/30 | Train: 2022-12-28 → 2025-12-11 | Test: 2025-12-15 → 2025-12-19 | Train_n=735 | Test_n=5


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 2/30 | Train: 2022-12-20 → 2025-12-04 | Test: 2025-12-08 → 2025-12-12 | Train_n=735 | Test_n=5


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 3/30 | Train: 2022-12-13 → 2025-11-25 | Test: 2025-12-01 → 2025-12-05 | Train_n=735 | Test_n=5


...

Run 27/30 | Train: 2018-07-13 → 2025-05-29 | Test: 2025-06-06 → 2025-06-12 | Train_n=1715 | Test_n=5
Run 28/30 | Train: 2018-07-06 → 2025-05-21 | Test: 2025-05-30 → 2025-06-05 | Train_n=1715 | Test_n=5


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 29/30 | Train: 2018-06-27 → 2025-05-14 | Test: 2025-05-22 → 2025-05-29 | Train_n=1715 | Test_n=5


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Pyth

Run 30/30 | Train: 2018-06-20 → 2025-05-07 | Test: 2025-05-15 → 2025-05-21 | Train_n=1715 | Test_n=5


In [11]:
results_df.to_csv('test_output.csv')